# 01 Data preprocessing notebook

## Import packages

In [1]:
from pathlib import Path
import json
from collections import Counter
import pandas as pd

## Set dataset paths

In [2]:
PROJECT_ROOT = Path("E:/Be_My_Ear")

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "ASL"
VIDEOS_DIR = RAW_DIR / "videos"
META_FILE = RAW_DIR / "nslt_100.json"
CLASS_LIST_FILE = RAW_DIR / "wlasl_class_list.txt"

print("Raw folder:", RAW_DIR)
print("Videos folder exists:", VIDEOS_DIR.exists())
print("Metadata file exists:", META_FILE.exists())
print("Class list exists:", CLASS_LIST_FILE.exists())

Raw folder: E:\Be_My_Ear\data\raw\ASL
Videos folder exists: True
Metadata file exists: True
Class list exists: True


## Load class name


In [3]:
class_names = {}

with open(CLASS_LIST_FILE, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            class_id = int(parts[0])
            gloss = parts[1]
            class_names[class_id] = gloss

print("Total class names:", len(class_names))
list(class_names.items())[:10]

Total class names: 2000


[(0, 'book'),
 (1, 'drink'),
 (2, 'computer'),
 (3, 'before'),
 (4, 'chair'),
 (5, 'go'),
 (6, 'clothes'),
 (7, 'who'),
 (8, 'candy'),
 (9, 'cousin')]

## Load WLASL100 metadata

In [4]:
with open(META_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total metadata entries:", len(data))

first_key = list(data.keys())[0]
print("First video ID:", first_key)
print("First item:", data[first_key])

Total metadata entries: 2038
First video ID: 05237
First item: {'subset': 'train', 'action': [77, 1, 55]}


## Check existing and missing videos

In [5]:
records = []

for video_id, item in data.items():
    class_id = item["action"][0]
    gloss = class_names.get(class_id, "unknown")

    video_path = VIDEOS_DIR / f"{video_id}.mp4"
    exists = video_path.exists()

    records.append({
        "video_id": video_id,
        "class_id": class_id,
        "gloss": gloss,
        "video_path": str(video_path),
        "exists": exists
    })

df = pd.DataFrame(records)
df.head()

,video_id,class_id,gloss,video_path,exists
0,05237,77,basketball,E:\Be_My_Ear\data\raw\ASL\videos\05237.mp4,False
1,69422,27,orange,E:\Be_My_Ear\data\raw\ASL\videos\69422.mp4,True
2,10899,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10899.mp4,False
3,10898,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10898.mp4,True
4,10893,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10893.mp4,True


## Dataset summary


In [6]:
print("Total videos in metadata:", len(df))
print("Existing videos:", df["exists"].sum())
print("Missing videos:", (~df["exists"]).sum())
print("Total classes:", df["class_id"].nunique())

df["gloss"].value_counts().head(10)

Total videos in metadata: 2038
Existing videos: 1013
Missing videos: 1025
Total classes: 100


gloss
book        40
drink       35
computer    30
before      26
go          26
chair       26
who         25
clothes     25
candy       24
deaf        23
Name: count, dtype: int64

In [8]:
#Keep only videos that exist
df_existing = df[df["exists"] == True].copy()

print("Usable videos:", len(df_existing))
df_existing.head()

Usable videos: 1013


,video_id,class_id,gloss,video_path,exists
1,69422,27,orange,E:\Be_My_Ear\data\raw\ASL\videos\69422.mp4,True
3,10898,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10898.mp4,True
4,10893,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10893.mp4,True
5,10892,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10892.mp4,True
7,10895,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10895.mp4,True


## Save the usable video list

In [9]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path("E:/Be_My_Ear")

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "ASL"
VIDEOS_DIR = RAW_DIR / "videos"
META_FILE = RAW_DIR / "nslt_100.json"
CLASS_LIST_FILE = RAW_DIR / "wlasl_class_list.txt"

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / "WLASL100"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load class names
class_names = {}

with open(CLASS_LIST_FILE, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            class_id = int(parts[0])
            gloss = parts[1]
            class_names[class_id] = gloss

# Load metadata
with open(META_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

records = []

for video_id, item in data.items():
    class_id = item["action"][0]
    gloss = class_names.get(class_id, "unknown")

    video_path = VIDEOS_DIR / f"{video_id}.mp4"

    if video_path.exists():
        records.append({
            "video_id": video_id,
            "class_id": class_id,
            "gloss": gloss,
            "video_path": str(video_path)
        })

df = pd.DataFrame(records)

output_file = OUTPUT_DIR / "wlasl100_video_index.csv"
df.to_csv(output_file, index=False)

print("Saved usable video index to:")
print(output_file)

print("\nUsable videos:", len(df))
print("Classes:", df["class_id"].nunique())

df.head()

Saved usable video index to:
E:\Be_My_Ear\data\processed\ASL\WLASL100\wlasl100_video_index.csv

Usable videos: 1013
Classes: 100


,video_id,class_id,gloss,video_path
0,69422,27,orange,E:\Be_My_Ear\data\raw\ASL\videos\69422.mp4
1,10898,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10898.mp4
2,10893,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10893.mp4
3,10892,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10892.mp4
4,10895,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10895.mp4


## Save WLASL100 usable index


In [13]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path("E:/Be_My_Ear")

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "ASL"
VIDEOS_DIR = RAW_DIR / "videos"
META_FILE = RAW_DIR / "nslt_100.json"
CLASS_LIST_FILE = RAW_DIR / "wlasl_class_list.txt"

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / "WLASL100"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load class names
class_names = {}

with open(CLASS_LIST_FILE, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            class_id = int(parts[0])
            gloss = parts[1]
            class_names[class_id] = gloss

# Load WLASL100 metadata
with open(META_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

records = []

for video_id, item in data.items():
    original_class_id = item["action"][0]
    gloss = class_names.get(original_class_id, "unknown")

    video_path = VIDEOS_DIR / f"{video_id}.mp4"

    if video_path.exists():
        records.append({
            "video_id": video_id,
            "original_class_id": original_class_id,
            "gloss": gloss,
            "video_path": str(video_path)
        })

df_usable = pd.DataFrame(records)

# Create new label IDs from 0 to 99
glosses = sorted(df_usable["gloss"].unique())
gloss_to_label_id = {gloss: idx for idx, gloss in enumerate(glosses)}

df_usable["label_id"] = df_usable["gloss"].map(gloss_to_label_id)

output_file = OUTPUT_DIR / "wlasl100_video_index.csv"
df_usable.to_csv(output_file, index=False)

print("Saved WLASL100 usable video index to:")
print(output_file)

print("\nUsable videos:", len(df_usable))
print("Total classes:", df_usable["label_id"].nunique())

df_usable.head()

Saved WLASL100 usable video index to:
E:\Be_My_Ear\data\processed\ASL\WLASL100\wlasl100_video_index.csv

Usable videos: 1013
Total classes: 100


,video_id,original_class_id,gloss,video_path,label_id
0,69422,27,orange,E:\Be_My_Ear\data\raw\ASL\videos\69422.mp4,67
1,10898,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10898.mp4,20
2,10893,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10893.mp4,20
3,10892,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10892.mp4,20
4,10895,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10895.mp4,20


In [ ]:
#Check whether all 100 classes are usable
class_counts = df_usable["gloss"].value_counts()

print("Number of classes:", len(class_counts))
print("Minimum videos in one class:", class_counts.min())
print("Maximum videos in one class:", class_counts.max())
print("Average videos per class:", class_counts.mean())

print("\nClasses with fewer than 5 videos:")
print(class_counts[class_counts < 5])

class_counts.head(20)

Number of classes: 100
Minimum videos in one class: 5
Maximum videos in one class: 16
Average videos per class: 10.13

Classes with fewer than 5 videos:
Series([], Name: count, dtype: int64)


gloss
before          16
cool            16
thin            16
drink           15
go              15
cousin          14
who             14
computer        14
help            14
tall            13
candy           13
thanksgiving    13
bed             13
accident        13
bowling         13
short           13
yes             12
basketball      12
shirt           12
dark            12
Name: count, dtype: int64